In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.manual_seed(42)

In [3]:
names = [
    "ANAS", "ANNA", "ARJUN", "ARUN", "AMIR",
    "MAYA", "MIRA", "MEERA", "ZARA", "ZAYAN",
    "RAHUL", "ROHAN", "RIYA", "REHAN", "RAYAN",
    "ADAM", "AHMED", "AYAAN", "ALI", "ALINA",
    "NINA", "NORA", "LEO", "LINA", "SARA",
    "SAM", "TARA", "OMAR", "KIRAN", "NOAH"
]

# Special tokens mark the start and end of a name.
special_chars = ["^", "$"]

# One entry per unique character.
chars = sorted(set("".join(names)) | set(special_chars))

char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}

vocab_size = len(chars)

print("Vocabulary:", chars)
print("Vocabulary size:", vocab_size)
print("Character to index mapping:", char_to_idx)
print("Index to character mapping:", idx_to_char)
print("ID of A:", char_to_idx["A"])

Vocabulary: ['$', 'A', 'D', 'E', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'R', 'S', 'T', 'U', 'Y', 'Z', '^']
Vocabulary size: 19
Character to index mapping: {'$': 0, 'A': 1, 'D': 2, 'E': 3, 'H': 4, 'I': 5, 'J': 6, 'K': 7, 'L': 8, 'M': 9, 'N': 10, 'O': 11, 'R': 12, 'S': 13, 'T': 14, 'U': 15, 'Y': 16, 'Z': 17, '^': 18}
Index to character mapping: {0: '$', 1: 'A', 2: 'D', 3: 'E', 4: 'H', 5: 'I', 6: 'J', 7: 'K', 8: 'L', 9: 'M', 10: 'N', 11: 'O', 12: 'R', 13: 'S', 14: 'T', 15: 'U', 16: 'Y', 17: 'Z', 18: '^'}
ID of A: 1


In [4]:
class NameGenerator(nn.Module):
    def __init__(self, vocab_size, embedding_dim=16, hidden_dim=64):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(
            embedding_dim,
            hidden_dim,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        x = self.embedding(x)
        out, hidden = self.rnn(x, hidden)
        logits = self.fc(out)
        return logits, hidden

    def init_hidden(self, device):
        return torch.zeros(1, 1, self.hidden_dim, device=device)

model = NameGenerator(vocab_size=vocab_size)
print(model)

NameGenerator(
  (embedding): Embedding(19, 16)
  (rnn): RNN(16, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=19, bias=True)
)


In [5]:
name = "ANAS"
input_text = "^" + name + "$"
target_text = name + "$"
x = torch.tensor(
    [[char_to_idx[ch] for ch in input_text]],
    dtype=torch.long
)

x, x.shape

(tensor([[18,  1, 10,  1, 13,  0]]), torch.Size([1, 6]))

In [6]:
input_text, target_text

('^ANAS$', 'ANAS$')

In [7]:
y = torch.tensor(
    [[char_to_idx[ch] for ch in target_text]],
    dtype=torch.long
)
y.shape

torch.Size([1, 5])

In [10]:
optimizer = optim.Adam(model.parameters(), lr=0.005)
epochs = 5000

for epoch in range(epochs):
    total_loss = 0.0

    for name in names:
        input_text = "^" + name
        target_text = name + "$"

        x = torch.tensor(
            [[char_to_idx[ch] for ch in input_text]],
            dtype=torch.long
        )

        y = torch.tensor(
            [[char_to_idx[ch] for ch in target_text]],
            dtype=torch.long
        )

        optimizer.zero_grad()

        logits, hidden = model(x)

        loss = F.cross_entropy(
            logits.reshape(-1, vocab_size),
            y.reshape(-1)
        )

        loss.backward()

        # Optional: limit unusually large gradients
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(names)

    if (epoch + 1) % 50 == 0:
        print(
            f"Epoch {epoch + 1}/{epochs}, "
            f"Average Loss: {avg_loss:.4f}"
        )

Epoch 50/5000, Average Loss: 0.7166
Epoch 100/5000, Average Loss: 0.7003
Epoch 150/5000, Average Loss: 0.6971
Epoch 200/5000, Average Loss: 0.6974
Epoch 250/5000, Average Loss: 0.6963
Epoch 300/5000, Average Loss: 0.6978
Epoch 350/5000, Average Loss: 0.6968
Epoch 400/5000, Average Loss: 0.6961
Epoch 450/5000, Average Loss: 0.6969
Epoch 500/5000, Average Loss: 0.6964
Epoch 550/5000, Average Loss: 0.6972
Epoch 600/5000, Average Loss: 0.7029
Epoch 650/5000, Average Loss: 0.6973
Epoch 700/5000, Average Loss: 0.6965
Epoch 750/5000, Average Loss: 0.6973
Epoch 800/5000, Average Loss: 0.6953
Epoch 850/5000, Average Loss: 0.6959
Epoch 900/5000, Average Loss: 0.6948
Epoch 950/5000, Average Loss: 0.6954
Epoch 1000/5000, Average Loss: 0.7005
Epoch 1050/5000, Average Loss: 0.6960
Epoch 1100/5000, Average Loss: 0.6966
Epoch 1150/5000, Average Loss: 0.6958
Epoch 1200/5000, Average Loss: 0.6952
Epoch 1250/5000, Average Loss: 0.6945
Epoch 1300/5000, Average Loss: 0.6969
Epoch 1350/5000, Average Loss: 0

In [17]:

@torch.no_grad()
def generate_name(
    model,
    temperature=0.8,
    max_length=15,
    starting_char="^"
):
    # Disable gradient calculation.
    # We are generating text, not training the model.
    # This saves memory and computation.
    model.eval()

    # Step 1: Start with the special start token "^".
    # The model learned during training that "^" means
    # a new name is beginning.
    generated = "^"

    # Step 2: If the user provides a starting letter,
    # append it to "^".
    #
    # Example:
    # starting_char = "A"
    # generated becomes "^A".
    #
    # If starting_char is "^", do not append anything.
    if starting_char != "^":
        generated += starting_char

    # Step 3: Generate one character per iteration.
    # max_length limits how many characters we generate
    # after the initial prefix.
    for _ in range(max_length):

        # Step 4: Convert the generated characters into
        # integer IDs because nn.Embedding expects indices.
        #
        # Example:
        # generated = "^A"
        # char_to_idx["^"] = 0
        # char_to_idx["A"] = 3
        # x = tensor([[0, 3]])
        #
        # Shape: (batch_size=1, sequence_length=2)
        x = torch.tensor(
            [[char_to_idx[ch] for ch in generated]],
            dtype=torch.long
        )

        # Step 5: Pass the entire current prefix through
        # the embedding layer and RNN, then the linear layer.
        #
        # logits shape:
        # (1, sequence_length, vocab_size)
        #
        # The model produces next-character scores
        # at every position in the prefix.
        logits, _ = model(x)

        # Step 6: Select the scores from the LAST position.
        #
        # logits[0, -1, :] means:
        #   0  -> first item in the batch
        #  -1  -> last character position
        #   :  -> scores for every vocabulary character
        #
        # clone() creates a separate copy so we can modify it.
        next_logits = logits[0, -1, :].clone()

        # Step 7: Prevent "^" from being generated again.
        # "^" is only meant to mark the beginning of a name.
        #
        # Setting its score to -infinity makes its
        # probability zero after softmax.
        next_logits[char_to_idx["^"]] = float("-inf")

        # Step 8: Convert scores into probabilities.
        #
        # Softmax converts arbitrary scores into probabilities
        # that sum to 1.
        #
        # Dividing by temperature controls randomness:
        # lower temperature -> more concentrated probabilities
        # higher temperature -> more spread-out probabilities
        probabilities = F.softmax(
            next_logits / temperature,
            dim=-1
        )

        # Step 9: Randomly sample ONE character ID
        # according to the predicted probabilities.
        #
        # Unlike argmax(), which always chooses the most
        # probable character, multinomial() can choose
        # another character with a lower probability.
        #
        # .item() converts the single tensor value into
        # a regular Python integer.
        next_id = torch.multinomial(
            probabilities,
            num_samples=1
        ).item()

        # Step 10: Convert the sampled integer ID back
        # into its character.
        #
        # Example:
        # next_id = 5
        # next_char = "N"
        next_char = idx_to_char[next_id]

        # Step 11: Check whether the model predicted "$".
        #
        # During training, "$" was the target after the
        # last character of a name. It means "name finished".
        #
        # If "$" is predicted, stop generating.
        # Do not include "$" in the returned name.
        if next_char == "$":
            break

        # Step 12: Append the new character to the prefix.
        #
        # Example:
        # generated = "^A"
        # next_char = "N"
        # generated becomes "^AN"
        #
        # The next iteration will use "^AN" as input.
        generated += next_char

    # Step 13: Restore training mode.
    # This is useful if you want to train the model again
    # after generation. For your current RNN, eval() does
    # not change the output because it has no dropout.
    model.train()

    # Step 14: Remove the initial "^" and return the name.
    #
    # Example:
    # "^ANAS" -> "ANAS"
    return generated[1:]


In [18]:

# Step 15: Generate 10 names, each starting with "A".
for _ in range(10):
    print(generate_name(model, starting_char="A"))


ALINA
ARJUN
AHMED
ALI
ADAM
ADAM
ARUN
ANNA
ADAM
ADAM
